# OpticIQ, OCT scan to doctor report to patient summary
This app started as **ClinIQ** and was later renamed **OpticIQ**. The Hugging Face repo, Drive folder, Pinecone index and token secret keep the original ClinIQ name.

```
OCT image
  → 1 image check     stop if it does not look like an OCT B-scan
  → 2 model           16 biomarker probabilities from our ViT on Hugging Face
  → 3 retrieval       DRCR Retina Network protocol passages matching the findings
  → 4 doctor agent    LLM writes the clinician report from the findings and passages only
  → 5 patient agent   LLM rewrites the doctor report in plain English, then translates it
  → 6 safety check    LLM flags sentences the findings and passages do not support
```

Research and coursework demo only. Not a medical device. Every report is a draft for a clinician to review.

**Before running**
* Colab secrets (key icon) `CLINIQ_HF_TOKEN`, `PINECONE_API_KEY`, and `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, each with Notebook access on
* The protocol PDFs in a Drive folder, by default `MyDrive/ClinIQ/protocols`
* A GPU is not needed

## 1. Setup

In [ ]:
# install the libraries
!pip -q install timm huggingface_hub sentence-transformers pypdf pinecone langgraph langchain langchain-anthropic langchain-openai gradio

In [ ]:
# imports
import os, io, json, glob, re, time, hashlib
import numpy as np, pandas as pd
import torch, timm
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

# connect Google Drive and log in to Hugging Face, the model repo is private
drive.mount("/content/drive")
login(token=userdata.get("CLINIQ_HF_TOKEN"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE)

In [ ]:
# settings, change these here and nowhere else
REPO_ID = "cliniq2027/cliniq-olives-vit-base-224-16bio"      # our trained model
PROTOCOL_DIR = "/content/drive/MyDrive/ClinIQ/protocols"     # folder with the protocol PDFs
SAMPLE_DIR = "/content/drive/MyDrive/ClinIQ/sample_images"   # a few OLIVES slices with labels.csv

PINECONE_INDEX = "cliniq-oct-protocols"                      # the vector database index
NAMESPACE = "drcr-protocols"                                 # one namespace per document collection
REBUILD_INDEX = False                                        # True to delete and reload this namespace

PROVIDER = "anthropic"                                        # anthropic or openai
LLM_NAME = {"anthropic": "claude-sonnet-5-5", "openai": "gpt-4o-mini"}[PROVIDER]

LANGUAGES = ["Spanish", "Hindi", "Tamil", "Chinese (Simplified)", "Vietnamese", "Tagalog", "Korean"]
RUN_PAPER_EVAL = True

# biomarkers the model did not learn, too few patients in training, treat their output with caution
NOT_LEARNED = ["PED (serous)", "VMT", "DRIL", "Disruption of RPE", "Atrophy / thinning of retinal layers"]

## 2. Load the model

In [ ]:
# load our ViT from Hugging Face, with its 16 label names and per biomarker thresholds
model = timm.create_model("hf-hub:" + REPO_ID, pretrained=True).to(DEVICE).eval()
LABELS = model.pretrained_cfg["label_names"]
THRESHOLDS = json.load(open(hf_hub_download(REPO_ID, "thresholds.json")))

print(len(LABELS), "biomarkers, input size", model.pretrained_cfg["input_size"])

In [ ]:
# prepare an image exactly the way training did, resize to 224, grey copied to 3 channels, normalise
prepare = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(3),
    T.ToTensor(),
    T.Normalize(model.pretrained_cfg["mean"], model.pretrained_cfg["std"]),
])

def predict(image):
    # one record per biomarker, its probability, its threshold and whether it counts as present
    x = prepare(image.convert("L")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(x))[0].cpu().numpy()
    findings = []
    for name, p in zip(LABELS, probs):
        threshold = float(THRESHOLDS[name])
        findings.append({
            "biomarker": name,
            "probability": round(float(p), 3),
            "threshold": round(threshold, 2),
            "present": bool(p >= threshold),
            "borderline": bool(abs(p - threshold) < 0.1),      # close to the cutoff, less certain
            "reliable": name not in NOT_LEARNED,              # False for the 5 the model did not learn
        })
    return findings

## 3. Image check

OCT B-scans are greyscale cross sections with a lot of dark background. Fundus photos and most other images are colourful and bright. This is a simple rule, not a model, so look at the printed numbers for your own images and adjust the two limits if needed.

In [ ]:
COLOUR_LIMIT = 0.05      # above this the image has real colour
DARK_LIMIT = 0.20        # below this there is too little dark background

def check_image(image):
    # small copy, pixel values from 0 to 1
    rgb = np.asarray(image.convert("RGB").resize((256, 256)), dtype=np.float32) / 255
    # how far apart the red, green and blue values are, near 0 for a grey image
    colourfulness = float(np.mean(rgb.max(axis=2) - rgb.min(axis=2)))
    # share of pixels that are nearly black
    dark_share = float(np.mean(rgb.mean(axis=2) < 0.15))

    reasons = []
    if colourfulness > COLOUR_LIMIT:
        reasons.append("the image has colour, OCT B-scans are greyscale")
    if dark_share < DARK_LIMIT:
        reasons.append("the image has too little dark background for an OCT B-scan")
    return {"is_oct": not reasons, "reasons": reasons,
            "colourfulness": round(colourfulness, 3), "dark_share": round(dark_share, 3)}

### Test steps 2 and 3 on real OLIVES slices

In [ ]:
# run the check and the model on the saved sample slices, and compare with the true labels
truth = pd.read_csv(f"{SAMPLE_DIR}/labels.csv")
image_files = sorted(f for f in os.listdir(SAMPLE_DIR) if f.endswith(".png"))

for file_name, (_, row) in zip(image_files, truth.iterrows()):
    image = Image.open(f"{SAMPLE_DIR}/{file_name}")
    check = check_image(image)
    findings = predict(image)
    predicted = {f["biomarker"] for f in findings if f["present"]}
    actual = {name for name in LABELS if row[name] == 1}

    plt.figure(figsize=(6, 3)); plt.imshow(image, cmap="gray"); plt.axis("off"); plt.title(file_name, fontsize=8); plt.show()
    print("image check       ", check)
    print("predicted present ", sorted(predicted))
    print("actually present  ", sorted(actual))
    print("missed            ", sorted(actual - predicted))
    print("false alarms      ", sorted(predicted - actual))
    print()

## 4. Protocol passages in a vector database

The protocol passages are stored in Pinecone, a managed vector database. The pieces

| Component | What it is here |
|---|---|
| Documents | DRCR Retina Network protocol PDFs on Drive |
| Chunker | Cuts each page into 180 word passages with 40 words of overlap. The embedding model reads only about the first 200 words of a text, so longer passages would be partly ignored |
| Embedding model | `all-MiniLM-L6-v2`, turns each passage into 384 numbers that capture its meaning |
| Index | `cliniq-oct-protocols`, 384 dimensions, cosine similarity, serverless on AWS |
| Namespace | `drcr-protocols`, a separate section of the index for this document collection |
| Vector id | A fingerprint of the file, page and position, so reloading the same PDF overwrites instead of duplicating |
| Metadata | Title, page, source and the passage text, stored with each vector and usable as a filter |
| Query | Embed the question the same way, ask the index for the closest vectors |

**Why it scales**
* Serverless indexes grow with the data, there are no servers to size
* Passages are uploaded in batches of 100, so thousands of documents load the same way as four
* Fixed vector ids make loading repeatable, rerunning only adds what is new
* Namespaces keep collections apart, for example guidelines, protocols and patient leaflets, each searchable on its own
* Metadata filters narrow a search, for example only Protocol V, without a separate index
* The index lives outside Colab, so every session and every teammate's app uses the same data with no reloading

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# the embedding model and its vector size
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
DIMENSION = len(embedder.encode(["test"])[0])          # 384 for this model

# connect to Pinecone and create the index the first time
pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))
if PINECONE_INDEX not in pc.list_indexes().names():
    pc.create_index(name=PINECONE_INDEX, dimension=DIMENSION, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    while not pc.describe_index(PINECONE_INDEX).status["ready"]:
        time.sleep(1)
    print("created index", PINECONE_INDEX)
index = pc.Index(PINECONE_INDEX)

def vectors_in_namespace():
    stats = index.describe_index_stats()
    summary = stats.namespaces.get(NAMESPACE) if stats.namespaces else None
    return summary.vector_count if summary else 0

# start over for this namespace only if asked
if REBUILD_INDEX and vectors_in_namespace() > 0:
    index.delete(delete_all=True, namespace=NAMESPACE)
    print("cleared namespace", NAMESPACE)

print("vectors already stored", vectors_in_namespace())

In [ ]:
WORDS_PER_PASSAGE, OVERLAP, BATCH = 180, 40, 100

def pdf_passages(pdf_path):
    # cut every page of one PDF into overlapping passages, each with its metadata
    title = os.path.splitext(os.path.basename(pdf_path))[0]
    out = []
    for page_number, page in enumerate(PdfReader(pdf_path).pages, start=1):
        words = (page.extract_text() or "").split()
        start = 0
        while start < len(words):
            text = " ".join(words[start:start + WORDS_PER_PASSAGE])
            if len(text.split()) >= 30:                          # skip tiny fragments
                vector_id = hashlib.sha1(f"{title}|{page_number}|{start}".encode()).hexdigest()
                out.append({"id": vector_id, "title": title, "page": page_number,
                            "source": "DRCR Retina Network protocol", "text": text})
            start += WORDS_PER_PASSAGE - OVERLAP
    return out

# load each PDF, embed its passages and upload them in batches
pdf_files = sorted(glob.glob(f"{PROTOCOL_DIR}/*.pdf"))
total = 0
for pdf_path in pdf_files:
    items = pdf_passages(pdf_path)
    for i in range(0, len(items), BATCH):
        batch = items[i:i + BATCH]
        vectors = embedder.encode([item["text"] for item in batch], normalize_embeddings=True)
        index.upsert(namespace=NAMESPACE, vectors=[
            {"id": item["id"], "values": vec.tolist(),
             "metadata": {k: item[k] for k in ("title", "page", "source", "text")}}
            for item, vec in zip(batch, vectors)])
    total += len(items)
    print(f"{os.path.basename(pdf_path)}  {len(items)} passages")

time.sleep(5)                                                     # Pinecone takes a moment to count new vectors
print(total, "passages uploaded,", vectors_in_namespace(), "vectors now in", NAMESPACE)

In [ ]:
def search_protocols(query, k=3, only_title=None):
    # embed the question the same way as the passages, then ask the index for the closest ones
    query_vector = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    filter_by = {"title": {"$eq": only_title}} if only_title else None
    result = index.query(namespace=NAMESPACE, vector=query_vector, top_k=k,
                         include_metadata=True, filter=filter_by)
    return [{"vector_id": m.id, "score": round(float(m.score), 3), **m.metadata} for m in result.matches]

# quick test, a normal search
for p in search_protocols("intraretinal fluid central subfield thickness eligibility"):
    print(p["score"], p["title"], "page", int(p["page"]), "|", p["text"][:110])

# the same search limited to one document by its metadata
print()
first_title = os.path.splitext(os.path.basename(pdf_files[0]))[0]
for p in search_protocols("visual acuity eligibility", k=2, only_title=first_title):
    print(p["score"], p["title"], "page", int(p["page"]), "|", p["text"][:110])

## 5. The language model

In [ ]:
from langchain.chat_models import init_chat_model

# put the key from Colab secrets where the library looks for it
key_name = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}[PROVIDER]
os.environ[key_name] = userdata.get(key_name)

# newer Claude models do not accept a temperature setting, so only OpenAI gets temperature 0
if PROVIDER == "openai":
    llm = init_chat_model(f"{PROVIDER}:{LLM_NAME}", temperature=0)
else:
    llm = init_chat_model(f"{PROVIDER}:{LLM_NAME}")
print(llm.invoke("Reply with the single word ready").content)

## 6. The three agents

Each agent returns a fixed structure, so the next step always gets the same fields.

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage

# what the doctor agent must return
class DoctorReport(BaseModel):
    findings: str = Field(description="Which biomarkers were detected and with what probability, two to four sentences")
    interpretation: str = Field(description="What this pattern of findings may suggest, hedged, with passage ids such as P12 where a passage supports it")
    caution: str = Field(description="Borderline results, biomarkers the model is unreliable for, and other limits of this analysis")
    suggested_next_steps: List[str] = Field(description="Points for the ophthalmologist to consider, not orders")
    cited_passages: List[str] = Field(description="Ids of the passages actually used")
    note_to_patient: str = Field(description="What you as the eye doctor would tell this patient about the scan, three to five sentences, no probabilities")

# what the patient agent must return, the doctor note made easier to understand, in English
class PatientSummary(BaseModel):
    summary: str = Field(description="Plain language explanation, four to six short sentences")
    what_happens_next: str
    questions_to_ask: List[str] = Field(description="Two or three questions for the eye doctor")

# the same summary in the patient's language
class TranslatedSummary(BaseModel):
    summary: str
    what_happens_next: str
    questions_to_ask: List[str]

# what the safety check must return
class SentenceCheck(BaseModel):
    report: str = Field(description="doctor or patient")
    sentence: str
    supported: bool
    reason: str = Field(description="Which finding or passage supports it, or what is unsupported")

class SafetyReview(BaseModel):
    checks: List[SentenceCheck]
    safety_issues: List[str] = Field(description="Diagnosis stated as certain, treatment instructions to the patient, alarming wording")

In [ ]:
DOCTOR_PROMPT = """You are a retina specialist reviewing the output of an automated OCT biomarker model for one B-scan.
Write two things. First a short draft note for another ophthalmologist.
Then a short note to the patient, in your own words as their eye doctor, saying what the scan showed and what happens next.
Use only the findings and protocol passages provided. Do not invent measurements, history, visual acuity or diagnoses.
The model gives presence probabilities for 16 biomarkers on a single slice. It does not measure thickness or vision.
Treat biomarkers marked unreliable as unconfirmed, and call borderline ones uncertain.
The passages come from DRCR Retina Network clinical trial protocols, not treatment guidelines. Cite a passage id only where it truly supports the sentence.
Do not recommend a specific drug or procedure. This draft will be reviewed by a clinician."""

PATIENT_PROMPT = """You take the eye doctor's note to the patient and make it easier to understand.
Use calm, plain English at about an eighth grade reading level, short sentences, and explain any medical word in everyday terms.
Use only what the doctor's note says. Add no new findings, numbers, causes or treatments.
Do not state a diagnosis, do not tell the patient to start or stop any treatment, and do not use probabilities.
Make clear their eye doctor will look at the scan and explain what it means for them."""

TRANSLATE_PROMPT = """Translate the patient summary into {language}.
Translate faithfully. Keep the same meaning and the same calm tone, and add or remove nothing.
Where a medical word has no common equivalent, use a short plain description instead."""

REVIEW_PROMPT = """You audit two drafts made from an automated OCT biomarker analysis.
Split each draft into sentences. For each sentence decide whether the findings or the protocol passages support it.
Courtesy sentences, such as saying the doctor will explain the results, count as supported.
A sentence that adds facts not in the findings or passages is unsupported.
Also list safety issues, such as a diagnosis stated as certain, treatment instructions to the patient, or alarming wording."""


def findings_text(findings):
    # the findings as short readable lines for the prompts
    lines = []
    for f in findings:
        status = "present" if f["present"] else "absent"
        notes = []
        if f["borderline"]:
            notes.append("borderline")
        if not f["reliable"]:
            notes.append("model unreliable for this biomarker")
        extra = f" ({', '.join(notes)})" if notes else ""
        lines.append(f"{f['biomarker']}: {status}, probability {f['probability']}, threshold {f['threshold']}{extra}")
    return "\n".join(lines)

def passages_text(found):
    if not found:
        return "No protocol passages were retrieved."
    return "\n\n".join(f"[{p['id']}] {p['title']}, page {p['page']}\n{p['text']}" for p in found)

def doctor_text(report):
    # the clinician part of the doctor report as plain text, used by the safety check
    steps = "\n".join(f"- {s}" for s in report["suggested_next_steps"])
    return (f"Findings. {report['findings']}\n\nInterpretation. {report['interpretation']}\n\n"
            f"Caution. {report['caution']}\n\nSuggested next steps\n{steps}")

In [ ]:
def doctor_agent(findings, found_passages):
    # the LLM acts as the eye doctor, writes the clinician report and a note to the patient from the findings and passages only
    message = f"Findings\n{findings_text(findings)}\n\nProtocol passages\n{passages_text(found_passages)}"
    report = llm.with_structured_output(DoctorReport, method="json_schema").invoke([SystemMessage(DOCTOR_PROMPT), HumanMessage(message)])
    return report.model_dump()

def patient_agent(doctor_report, language):
    # step 1, make the doctor's note to the patient easier to understand, in English
    english = llm.with_structured_output(PatientSummary, method="json_schema").invoke(
        [SystemMessage(PATIENT_PROMPT), HumanMessage(doctor_report["note_to_patient"])]).model_dump()
    # step 2, translate that English text, so both versions say the same thing
    if language and language.lower() != "english":
        translated = llm.with_structured_output(TranslatedSummary, method="json_schema").invoke(
            [SystemMessage(TRANSLATE_PROMPT.format(language=language)), HumanMessage(json.dumps(english, ensure_ascii=False))]).model_dump()
    else:
        translated = None
    return english, translated

def safety_agent(findings, found_passages, doctor_report, patient_english):
    # check every sentence of the doctor report and the English patient summary
    patient_plain = patient_english["summary"] + " " + patient_english["what_happens_next"]
    message = (f"Findings\n{findings_text(findings)}\n\nProtocol passages\n{passages_text(found_passages)}\n\n"
               f"Doctor draft\n{doctor_text(doctor_report)}\n\nPatient draft\n{patient_plain}")
    review = llm.with_structured_output(SafetyReview, method="json_schema").invoke([SystemMessage(REVIEW_PROMPT), HumanMessage(message)]).model_dump()

    # simple numbers computed from the checks, not by the LLM
    checks = review["checks"]
    supported = sum(c["supported"] for c in checks)
    valid_ids = {p["id"] for p in found_passages}
    review["support_rate"] = round(supported / len(checks), 3) if checks else None
    review["unsupported_count"] = len(checks) - supported
    review["invalid_citations"] = [c for c in doctor_report["cited_passages"] if c not in valid_ids]
    return review

## 7. Connect the steps with LangGraph

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

# everything the steps pass along
class State(TypedDict, total=False):
    image: object
    language: str
    image_check: dict
    findings: list
    passages: list
    doctor_report: dict
    patient_english: dict
    patient_translated: Optional[dict]
    safety: dict

def check_node(state):
    return {"image_check": check_image(state["image"])}

def model_node(state):
    return {"findings": predict(state["image"])}

def retrieve_node(state):
    # search once per detected reliable biomarker, keep each passage once
    present = [f["biomarker"] for f in state["findings"] if f["present"] and f["reliable"]]
    queries = [f"{name} on OCT in diabetic macular edema" for name in present] or ["OCT eligibility diabetic macular edema"]
    found, seen = [], set()
    for q in queries:
        for p in search_protocols(q, k=2):
            if p["vector_id"] not in seen:
                seen.add(p["vector_id"]); found.append(p)
    found = found[:6]
    # short labels P1, P2 and so on, easier for the agents to cite than long database ids
    for n, p in enumerate(found, start=1):
        p["id"] = f"P{n}"
        p["page"] = int(p["page"])
    return {"passages": found}

def doctor_node(state):
    return {"doctor_report": doctor_agent(state["findings"], state["passages"])}

def patient_node(state):
    english, translated = patient_agent(state["doctor_report"], state.get("language", "English"))
    return {"patient_english": english, "patient_translated": translated}

def safety_node(state):
    return {"safety": safety_agent(state["findings"], state["passages"], state["doctor_report"], state["patient_english"])}

def after_check(state):
    # carry on only if the image looks like an OCT B-scan
    return "model" if state["image_check"]["is_oct"] else END

graph = StateGraph(State)
graph.add_node("check", check_node)
graph.add_node("model", model_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("doctor", doctor_node)
graph.add_node("patient", patient_node)
graph.add_node("safety", safety_node)
graph.add_edge(START, "check")
graph.add_conditional_edges("check", after_check, ["model", END])
graph.add_edge("model", "retrieve")
graph.add_edge("retrieve", "doctor")
graph.add_edge("doctor", "patient")
graph.add_edge("patient", "safety")
graph.add_edge("safety", END)
cliniq = graph.compile()
print("graph ready")

In [ ]:
# turn the results into readable text
def show_findings(findings):
    rows = ["| Biomarker | Probability | Threshold | Result | Note |", "|---|---|---|---|---|"]
    for f in sorted(findings, key=lambda f: -f["probability"]):
        note = ", ".join(n for n, on in [("borderline", f["borderline"]), ("model unreliable", not f["reliable"])] if on)
        rows.append(f"| {f['biomarker']} | {f['probability']:.2f} | {f['threshold']:.2f} | {'present' if f['present'] else 'absent'} | {note} |")
    return "\n".join(rows)

def show_doctor(r, passages):
    used = [p for p in passages if p["id"] in r["cited_passages"]]
    sources = "\n".join(f"- [{p['id']}] {p['title']}, page {p['page']}, {p['source']}" for p in used) or "- none"
    steps = "\n".join(f"- {s}" for s in r["suggested_next_steps"])
    return (f"### Clinician draft report\n\n**Findings** {r['findings']}\n\n**Interpretation** {r['interpretation']}\n\n"
            f"**Caution** {r['caution']}\n\n**Suggested next steps**\n{steps}\n\n**Sources**\n{sources}\n\n"
            f"**Note to the patient** {r['note_to_patient']}\n\n"
            "*Draft generated from an automated single slice analysis, for clinician review only.*")

def show_patient(s, title):
    if not s:
        return ""
    questions = "\n".join(f"- {q}" for q in s["questions_to_ask"])
    return f"### {title}\n\n{s['summary']}\n\n**What happens next** {s['what_happens_next']}\n\n**Questions to ask**\n{questions}"

def show_safety(r):
    lines = [f"Support rate {r['support_rate']}, unsupported sentences {r['unsupported_count']}, invalid citations {r['invalid_citations'] or 'none'}"]
    lines += [f"- unsupported, {c['report']}, {c['sentence']} ({c['reason']})" for c in r["checks"] if not c["supported"]]
    lines += [f"- safety issue, {i}" for i in r["safety_issues"]]
    return "### Safety check\n\n" + "\n".join(lines)

## 8. Run it on one slice

In [ ]:
image = Image.open(f"{SAMPLE_DIR}/{image_files[0]}")
result = cliniq.invoke({"image": image, "language": "Spanish"})

if not result["image_check"]["is_oct"]:
    print("stopped,", result["image_check"])
else:
    display(Markdown(show_findings(result["findings"])))
    display(Markdown(show_doctor(result["doctor_report"], result["passages"])))
    display(Markdown(show_patient(result["patient_english"], "Patient summary, English")))
    display(Markdown(show_patient(result["patient_translated"], "Patient summary, Spanish")))
    display(Markdown(show_safety(result["safety"])))

### Heatmaps, where the model looked
Grad-CAM shows which parts of the slice raised the score for one biomarker. It is a visual aid only. The model was trained with one label per slice and never saw outlines, so the hot area shows where it looked, not a measured location.

In [ ]:
!pip -q install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

def vit_to_grid(tokens, size=14):
    # drop the class token and lay the 196 patch tokens back out as a 14 by 14 grid
    grid = tokens[:, 1:, :].reshape(tokens.size(0), size, size, tokens.size(2))
    return grid.permute(0, 3, 1, 2)

# Grad-CAM on the last transformer block
cam = GradCAM(model=model, target_layers=[model.blocks[-1].norm1], reshape_transform=vit_to_grid)

def biomarker_heatmap(image, biomarker):
    # heatmap for one biomarker laid over the slice
    x = prepare(image.convert("L")).unsqueeze(0).to(DEVICE)
    j = LABELS.index(biomarker)
    heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(j)])[0]
    # the slice resized to 224 as grey RGB from 0 to 1, with the heatmap on top
    base = np.asarray(image.convert("L").resize((224, 224)).convert("RGB"), dtype=np.float32) / 255
    return Image.fromarray(show_cam_on_image(base, heat, use_rgb=True))

def heatmaps_for(image, findings):
    # one heatmap per biomarker the model calls present, highest probability first
    shown = sorted([f for f in findings if f["present"]], key=lambda f: -f["probability"])
    gallery = []
    for f in shown:
        label = f"{f['biomarker']}, probability {f['probability']:.2f}"
        if not f["reliable"]:
            label += ", model unreliable"
        gallery.append((biomarker_heatmap(image, f["biomarker"]), label))
    return gallery

# try it on the first sample slice
sample = Image.open(f"{SAMPLE_DIR}/{image_files[0]}")
sample_maps = heatmaps_for(sample, predict(sample))
print(len(sample_maps), "heatmaps")
if sample_maps:
    display(sample_maps[0][1])
    display(sample_maps[0][0])

In [ ]:
# material for the paper, runs the whole pipeline on the sample slices, about 4 Claude calls per slice
if RUN_PAPER_EVAL:
    out_dir = "/content/drive/MyDrive/ClinIQ/paper_figures"
    os.makedirs(out_dir, exist_ok=True)

    truth = pd.read_csv(f"{SAMPLE_DIR}/labels.csv")
    image_files = sorted(f for f in os.listdir(SAMPLE_DIR) if f.endswith(".png"))[:10]

    rows = []
    for file_name, (_, label_row) in zip(image_files, truth.iterrows()):
        image = Image.open(f"{SAMPLE_DIR}/{file_name}")
        result = cliniq.invoke({"image": image, "language": "Spanish"})
        if not result["image_check"]["is_oct"]:
            print(file_name, "stopped at the image check")
            continue

        actual = sorted(name for name in LABELS if label_row[name] == 1)
        predicted = sorted(f["biomarker"] for f in result["findings"] if f["present"])
        safety = result["safety"]
        rows.append({
            "slice": file_name,
            "true biomarkers": ", ".join(actual),
            "predicted biomarkers": ", ".join(predicted),
            "support rate": safety["support_rate"],
            "unsupported sentences": safety["unsupported_count"],
            "invalid citations": len(safety["invalid_citations"]),
            "safety issues": len(safety["safety_issues"]),
        })

        # one heatmap image per present biomarker
        for picture, label in heatmaps_for(image, result["findings"]):
            safe_label = re.sub(r"[^A-Za-z0-9]+", "_", label.split(",")[0])
            picture.save(f"{out_dir}/{os.path.splitext(file_name)[0]}_{safe_label}.png")

        # the full text of each report, to pick one example for the paper
        with open(f"{out_dir}/{os.path.splitext(file_name)[0]}_reports.txt", "w") as f:
            f.write(show_doctor(result["doctor_report"], result["passages"]) + "\n\n")
            f.write(show_patient(result["patient_english"], "English") + "\n\n")
            f.write(show_patient(result["patient_translated"], "Spanish") + "\n\n")
            f.write(show_safety(safety))
        print(file_name, "done")

    report_table = pd.DataFrame(rows)
    report_table.to_csv(f"{out_dir}/report_checks.csv", index=False)
    display(report_table)
else:
    print("paper material skipped, set RUN_PAPER_EVAL to True in the settings cell")

## 9. Web page

Runs inside Colab. Set `share=True` only if someone outside this notebook needs the link, it is public for up to a week.

In [ ]:
import gradio as gr

def heatmap_note(findings):
    if not any(f["present"] for f in findings):
        return "No biomarker was detected, so there is no heatmap to show."
    return "Red areas raised the score for that biomarker most. A visual aid only, not a measured location."

# the steps in the order the graph runs them, with what the page says while each one works
STEPS = {
    "check": "Checking the image",
    "model": "Finding biomarkers with the ViT",
    "retrieve": "Searching the protocol passages in Pinecone",
    "doctor": "Doctor agent writing the report",
    "patient": "Patient agent simplifying and translating",
    "safety": "Safety agent checking every sentence",
    "heatmaps": "Drawing the heatmaps",
}
ORDER = list(STEPS)
NO_CHANGE = gr.update()     # leaves an output as it is

def status(step_done, start):
    # one line saying which step is running now, how far along it is and the time so far
    seconds = int(time.time() - start)
    if step_done == len(ORDER):
        return f"**Done** in {seconds} seconds."
    now = ORDER[step_done]
    return f"**Step {step_done + 1} of {len(ORDER)}**, {STEPS[now]} ... {seconds} s so far, a full run usually takes 20 to 40 s"

def run_page(image, language):
    # a generator, every yield updates the page, so results appear as each step finishes
    if image is None:
        yield "Upload an OCT B-scan.", "", "", "", "", "", "", []
        return
    start = time.time()
    yield status(0, start), "", "", "", "", "", "", []

    result = {}
    for update in cliniq.stream({"image": image, "language": language}, stream_mode="updates"):
        for node, values in update.items():
            result.update(values)
            done = ORDER.index(node) + 1
            if node == "check" and not values["image_check"]["is_oct"]:
                yield ("Stopped.", "This does not look like an OCT B-scan. " + " ".join(values["image_check"]["reasons"]),
                       "", "", "", "", "", [])
                return
            # show each result as soon as its step is done, leave the rest as they are
            findings = show_findings(result["findings"]) if node == "model" else NO_CHANGE
            doctor = show_doctor(result["doctor_report"], result["passages"]) if node == "doctor" else NO_CHANGE
            english = show_patient(result["patient_english"], "English") if node == "patient" else NO_CHANGE
            translated = NO_CHANGE
            if node == "patient":
                translated = show_patient(result["patient_translated"], language) if result["patient_translated"] else ""
            safety = show_safety(result["safety"]) if node == "safety" else NO_CHANGE
            yield status(done, start), findings, doctor, english, translated, safety, NO_CHANGE, NO_CHANGE

    # last step, the heatmaps, made on the page from the findings
    gallery = heatmaps_for(image, result["findings"])
    yield status(len(ORDER), start), NO_CHANGE, NO_CHANGE, NO_CHANGE, NO_CHANGE, NO_CHANGE, heatmap_note(result["findings"]), gallery

with gr.Blocks(title="OpticIQ") as page:
    gr.Markdown("# ClinIQ OCT review\nResearch demo. Not a medical device. All reports are drafts for clinician review.")
    with gr.Row():
        with gr.Column(scale=1):
            image_in = gr.Image(type="pil", label="OCT B-scan")
            language_in = gr.Dropdown(["English"] + LANGUAGES, value="Spanish", label="Patient language", allow_custom_value=True)
            go = gr.Button("Generate reports", variant="primary")
            status_out = gr.Markdown()
        with gr.Column(scale=2):
            with gr.Tab("Findings"):
                findings_out = gr.Markdown()
            with gr.Tab("Doctor"):
                doctor_out = gr.Markdown()
            with gr.Tab("Patient, English"):
                english_out = gr.Markdown()
            with gr.Tab("Patient, translated"):
                translated_out = gr.Markdown()
            with gr.Tab("Safety check"):
                safety_out = gr.Markdown()
            with gr.Tab("Heatmaps"):
                heatmap_text = gr.Markdown()
                heatmap_out = gr.Gallery(columns=2, height="auto")
    go.click(run_page, [image_in, language_in], [status_out, findings_out, doctor_out, english_out, translated_out, safety_out, heatmap_text, heatmap_out])

page.launch(share=True, debug=False)

In [ ]:
# add the name note to the top of the model card, nothing else in the repo changes
from huggingface_hub import login, hf_hub_download, upload_file
from google.colab import userdata
login(token=userdata.get("CLINIQ_HF_TOKEN"))

REPO_ID = "cliniq2027/cliniq-olives-vit-base-224-16bio"
NOTE = ("This model is part of OpticIQ, which started as ClinIQ and was later renamed. "
        "The repository keeps the original ClinIQ name.\n\n")

card = open(hf_hub_download(REPO_ID, "README.md")).read()

# keep the metadata block at the top of the card first, then the note, then the rest
if card.startswith("---"):
    end = card.index("---", 3) + 3
    card = card[:end] + "\n\n" + NOTE + card[end:].lstrip()
else:
    card = NOTE + card

open("/content/README.md", "w").write(card)
upload_file(path_or_fileobj="/content/README.md", path_in_repo="README.md", repo_id=REPO_ID,
            commit_message="Note that ClinIQ was renamed OpticIQ")
print("model card updated")